# Nuova versione del driver della bilancia

In [ ]:
import pyserial

In [ ]:

def data_len(data):
    return len(data).to_bytes(1, byteorder='little')

def checksum(instr_code, data):
    lenght = data_len(data)

    # 1s complement of the message payload
    checksum = 255 - (sum(instr_code + lenght + data) % 256)

    return checksum.to_bytes(1, byteorder='little')

def build_message(comando, data=None):

    instr_code = comandi[comando]

    if data is not None:
        if comando == "Remote activation":
            data_encoded = sotto_comandi[comando][data]
        
        elif comando == "Config data-logging":

            byte1 = 0
            byte2 = 0

            for item in sotto_comandi[comando][0]:
                if item in data:
                    byte1 |= (1 << sotto_comandi[comando][0].index(item))

            for item in sotto_comandi[comando][1]:
                if item in data:
                    byte2 |= (1 << sotto_comandi[comando][1].index(item))

            data_encoded = bytes([byte1, byte2])

        else:
            raise ValueError("Comando non supportato o dati non validi")
    else:
        data_encoded = bytes()
    

    encoded_command = Header + dev_addr + instr_code + data_len(data_encoded) + data_encoded + checksum(instr_code, data_encoded)
    return encoded_command

def decode_message(message):
    header = message[0:2]
    address = message[2]
    instr_code = message[3]
    data_length = message[4]
    instr_code = message[5]
    data = message[6:6+data_length]
    received_checksum = message[6+data_length]

    return {
        "header": header,
        "address": address,
        "instr_code": instr_code,
        "data_length": data_length,
        "data": data,
        "received_checksum": received_checksum
    }


def read_from_buffer(ser, num_bytes):
    if ser.in_waiting >= num_bytes:
        message = ser.read(num_bytes)
        return message
    return None

def decode_ascii_data(data, sizes):
    split_message = []

    idx = 0
    for size in sizes:
        split_message.append(data[idx:idx+size].decode('ascii').strip())
        idx += size

    return split_message



In [ ]:
class Bilancia:
    Header = bytes([255, 254])
    dev_addr = bytes([1])

    comandi = {
        "Remote activation" : bytes([0]),
        "Send monitor config" : bytes([1]),
        "Send film parametes" : bytes([2]),
        "Receive film parameters" : bytes([3]),
        "Send monitor status" : bytes([4]),
        "Config data-logging" : bytes([5]),
    }

    sotto_comandi = {
        # vuole sempre 1 byte
        "Remote activation" : {
            "start" : bytes([1]),
            "stop" : bytes([2]),
            "shutter" : bytes([4]),
        },

        # vuole sempre 2 byte, fatti come descritto in seguito
        "Config data-logging" :
        [
            # byte 1
            ["Displayed rate", "Displayed thickness", "Displayed frequency", "Sensor 1 rate", "Sensor 1 thickness", "Sensor 1 frequency", "Sensor 2 rate", "Sensor 2 thickness"],
            # byte 2
            ["Sensor 2 frequency", "Active sensor number"]
        ]
    }

    def __init__(self, porta, baudrate=9600):
        self.ser = pyserial.Serial(porta, baudrate)
    
    def data_len(self, data):
        return len(data).to_bytes(1, byteorder='little')
    
    def checksum(self, instr_code, data):
        lenght = self.data_len(data)

        # 1s complement of the message payload
        checksum = 255 - (sum(instr_code + lenght + data) % 256)

        return checksum.to_bytes(1, byteorder='little')
    
    def build_message(self, comando, data=None):

        instr_code = self.comandi[comando]

        if data is not None:
            if comando == "Remote activation":
                data_encoded = self.sotto_comandi[comando][data]
            
            elif comando == "Config data-logging":

                byte1 = 0
                byte2 = 0

                for item in self.sotto_comandi[comando][0]:
                    if item in data:
                        byte1 |= (1 << self.sotto_comandi[comando][0].index(item))

                for item in self.sotto_comandi[comando][1]:
                    if item in data:
                        byte2 |= (1 << self.sotto_comandi[comando][1].index(item))

                data_encoded = bytes([byte1, byte2])

            else:
                raise ValueError("Comando non supportato o dati non validi")
        else:
            data_encoded = bytes()
        
        encoded_command = self.Header + self.dev_addr + instr_code + self.data_len(data_encoded) + data_encoded + self.checksum(instr_code, data_encoded)
        return encoded_command
    
    def decode_message(self, message):
        header = message[0:2]
        address = message[2]
        instr_code = message[3]
        data_length = message[4]
        instr_code = message[5]
        data = message[6:6+data_length]
        received_checksum = message[6+data_length]

        return {
            "header": header,
            "address": address,
            "instr_code": instr_code,
            "data_length": data_length,
            "data": data,
            "received_checksum": received_checksum
        }
    
    def read_from_buffer(self, num_bytes):
        if self.ser.in_waiting >= num_bytes:
            message = self.ser.read(num_bytes)
            return message
        return None
    
    def decode_ascii_data(self, data, sizes):
        split_message = []

        idx = 0
        for size in sizes:
            split_message.append(data[idx:idx+size].decode('ascii').strip())
            idx += size

        return split_message
    


    